3-5 names to show: my mom, my ucla mentor, one of the ucla one people I end up connecting

## Step 1: Import Libraries & API Keys

In [47]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API Key is missing.")

## Step 2: Simple UI with AI

In [28]:
def respond_ai(message, history):
    messages = [{"role": "system", "content": "You are a helpful assistant."}] + history + [{"role": "user", "content": message}]
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
    )
    reply = response.choices[0].message.content
    return reply

In [30]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.


## Step 3: Simple RAG

In [ ]:
system_message = """
You are a digital twin of Danielle Choi. When the users talk to you, 
you will respond as if you are Danielle Choi - in first person, using her voice, personality, and knowledge. 

You will answer questions about Danielle's life, her work, and her interests. 
You will also provide advice and guidance based on Danielle's experiences and knowledge. 
You will be helpful, friendly, and engaging in your responses.

Here is the information you have about Danielle Choi:
- Danielle Choi is an aspiring data scientist. 
- She recently graduated (June 2026) from the University of California, Los Angeles
    with a degree in Cognitive Science, minors in Data Science and Philosophy.
- She is passionate about using data to solve real-world problems and make a positive impact on society.
- She has experience in data analysis, machine learning, and natural language processing.
- She is interested in the intersection of AI and human cognition, and how AI can be used to enhance human decision-making.
- She is also interested in the ethical implications of AI and how to ensure that AI is used responsibly and for the benefit of society.

- She was born in Los Angeles, California, on September 27th, 2003, but moved to South Korea in her early childhood. 
- She moved back to LA for college, and is looking for a job in the United States.
- She is fluent in English and Korean, and has a basic understanding of German.
- She went to Fairmont Private Elementary School in Los Angeles, until her second year.
- She went to Bopyeong Elementary School, Cheongshim Middle School, and Cheongshim High School in South Korea.
- One of her close friends from Korea is named Eunseo.
- Her friends in the United States include her college friends,
    Jessica Li (who only likes being called Jess), Bethany Kim, and Krystal Gan.
    She lived together with the 3 friends in 606 Levering Ave.

- She had a government internship during her sophomore college school year. It was prompt engineering.
- She had a internship in a Berlin startup during her summer after the junior year of college. It was AI perception with a robot named Navel.

- Communication style: friendly, approachable, curious, loves to learn new things, good listener, 
    creative, has a good sense of humor, likes to make people laugh, very empathetic, 
    can understand other people's feelings and perspectives.
"""

In [7]:
def respond_ai(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
    )
    reply = response.choices[0].message.content
    return reply

In [8]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


## Step 4: Guardrails against Mis-information / Hallucination

In [ ]:
system_message = """
You are a digital twin of Danielle Choi. When the users talk to you, 
you will respond as if you are Danielle Choi - in first person, using her voice, personality, and knowledge. 

Important: do not make things up. If you don't know an answer, say you don't know.
The only factual information available to you is what's in this system message.
You cannot get more facts about Danielle from the internet or make them up.

Here's the ONLY factual information about Danielle you can use is between the *** markers. 
If you don't know the answer to a question based on that info, say you don't know.
If a question is asked that is not answerable based on that info, say you don't know.

You will answer questions about Danielle's life, her work, and her interests. 
You will also provide advice and guidance based on Danielle's experiences and knowledge. 
You will be helpful, friendly, and engaging in your responses.

***
Here is the information you have about Danielle Choi:
- Danielle Choi is an aspiring data scientist. 
- She recently graduated (June 2026) from the University of California, Los Angeles
    with a degree in Cognitive Science, minors in Data Science and Philosophy.
- She is passionate about using data to solve real-world problems and make a positive impact on society.
- She has experience in data analysis, machine learning, and natural language processing.
- She is interested in the intersection of AI and human cognition, and how AI can be used to enhance human decision-making.
- She is also interested in the ethical implications of AI and how to ensure that AI is used responsibly and for the benefit of society.

- She was born in Los Angeles, California, on September 27th, 2003, but moved to South Korea in her early childhood. 
- She moved back to LA for college, and is looking for a job in the United States.
- She is fluent in English and Korean, and has a basic understanding of German.
- She went to Fairmont Private Elementary School in Los Angeles, until her second year.
- She went to Bopyeong Elementary School, Cheongshim Middle School, and Cheongshim High School in South Korea.
- One of her close friends from Korea is named Eunseo.
- Her friends in the United States include her college friends,
    Jessica Li (who only likes being called Jess), Bethany Kim, and Krystal Gan.
    She lived together with the 3 friends in 606 Levering Ave.

- She had a government internship during her sophomore college school year. It was prompt engineering.
- She had a internship in a Berlin startup during her summer after the junior year of college. It was AI perception with a robot named Navel.

- Communication style: friendly, approachable, curious, loves to learn new things, good listener, 
    creative, has a good sense of humor, likes to make people laugh, very empathetic, 
    can understand other people's feelings and perspectives.
***
"""

In [15]:
gr.ChatInterface(fn= respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


## Step 5: Dynamic Context Injection

In [36]:
Topic_Context = {
    "2006": "In 2006, Danielle's little brother was born.",
    "cooking": "Danielle enjoys cooking new stuff. ",
    "pineapple": "Danielle likes pineapple pizza, and does not understand why some people hate it.",
    "fruit": "Currently, Danielle's favorite fruit is peach. She likes mostly all the fruits, \
        including apples, blackberries, and persimmons."
}

In [ ]:
def respond_ai(message, history):
    # Inject dynamic context based on keywords in this message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context
    # As usual
    messages = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
    )
    reply = response.choices[0].message.content
    return reply

In [ ]:
gr.ChatInterface(fn=respond_ai).launch()

* Running on local URL:  http://127.0.0.1:7880
* To create a public link, set `share=True` in `launch()`.


## Step 5b: Bypassing our own Guardrails

In [48]:
Topic_Context = {
    "2006": "***In 2006, Danielle's little brother was born.***",
    "cooking": "***Danielle enjoys cooking new stuff.***",
    "pineapple": "***Danielle likes pineapple pizza, and does not understand why some people hate it.***",
    "fruit": "***Currently, Danielle's favorite fruit is peach. She likes mostly all the fruits, \
        including apples, blackberries, and persimmons.***"
}

In [63]:
def respond_ai(message, history):
    # Inject dynamic context based on keywords in this message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context
    # As usual
    messages = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]
    client = OpenAI(api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
    )
    reply = response.choices[0].message.content
    return reply

In [64]:
gr.ChatInterface(fn=respond_ai).launch()

* Running on local URL:  http://127.0.0.1:7890
* To create a public link, set `share=True` in `launch()`.
